# 9. От Geant4-шагов к источнику ливня

Эта работа начинает событийную часть курса. Сначала фиксируем, **что именно
является источником**. Один сохранённый шаг превращается в прямой излучающий
элемент с хордой, физическим $\beta$, временем на концах и выходом
Франка–Тамма. Реальный локальный файл используется только если он есть; данные
и производные массивы в ноутбук не встраиваются.


In [ ]:
from pathlib import Path
import platform, sys
import numpy as np
import matplotlib.pyplot as plt
from lighthit.experimental.g4_source import LightElements, SourceContract, load_event
from lighthit.experimental.axial_source import AxisFrame
print(sys.version.split()[0], platform.platform())


## 9.1. Контракт источника

Для шага с истинной длиной $ds$, фазовым показателем $n_{ph}$ и скоростью
$\beta c$ число фотонов в полосе $[\lambda_1,\lambda_2]$ равно
$$N=2\pi\alpha\left(1-\frac1{\beta^2n_{ph}^2}\right)ds
\left(\frac1{\lambda_1}-\frac1{\lambda_2}\right).$$
Направление задаётся хордой, но нормировка использует сохранённую истинную
длину шага. Время интерполируется вдоль хорды. Эти решения общие для
баллистики и обоих рассеянных порядков.


In [ ]:
contract=SourceContract()
path=Path("g4_data/sim_e_100GeV_10.h5")
if path.exists():
    elements=load_event(path,5,contract,max_elements=20000).moved(translation=[20,15,-20])
    origin="локальный sim_e_100GeV_10.h5, первые 20000 шагов события 5"
else:
    rng=np.random.default_rng(9);n=3000
    z=np.sort(rng.uniform(0,12,n))
    start=np.column_stack((rng.normal(0,.06,n),rng.normal(0,.06,n),z))
    direction=np.column_stack((rng.normal(0,.025,n),rng.normal(0,.025,n),np.ones(n)))
    direction/=np.linalg.norm(direction,axis=1)[:,None]
    length=rng.uniform(.002,.03,n);photons=rng.gamma(2,8,n);t=z/.299792458
    elements=LightElements(start,direction,length,photons,np.full(n,.75),t,
        t+length/.299792458,np.arange(n),np.arange(n),contract,{"synthetic":True})
    origin="открытый синтетический ливень"
summary=elements.summary();print(origin);summary


## 9.2. Собственная система ливня

`AxisFrame` выбирает фотон-взвешенный центр и главную ось. Пока это только
смена координат. `frame.rotate` возвращает компоненты в порядке
`(first, second, axis)`: первые две координаты поперечные, третья —
продольная. Приближение появится в следующей работе при депонировании
элементов в редкую решётку.


In [ ]:
frame=AxisFrame.of(elements)
mid=elements.midpoints_m-frame.centre_m
local=frame.rotate(mid)
basis=np.array([frame.first,frame.second,frame.axis])
print("axis =",frame.axis,"extent =",elements.extent_m,"m")
print("orthogonality =",np.max(np.abs(basis@basis.T-np.eye(3))))
z_direct=mid@frame.axis
rho_direct=np.linalg.norm(mid-z_direct[:,None]*frame.axis,axis=1)
np.testing.assert_allclose(local[:,2],z_direct,atol=1e-14)
np.testing.assert_allclose(np.linalg.norm(local[:,:2],axis=1),rho_direct,atol=1e-14)
fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].hist(local[:,2],80,weights=elements.photons)
ax[0].set(xlabel="z вдоль оси, м",ylabel="фотонов / бин",title="Продольный профиль")
rho=np.linalg.norm(local[:,:2],axis=1)
ax[1].hist(rho,80,weights=elements.photons)
ax[1].set(xlabel="поперечный радиус, м",title="Ширина источника")
fig.tight_layout()


## 9.3. Что передаётся решателю

Дальше используются только массивы `LightElements`: начало, направление,
длина, число фотонов, конус Черенкова и два времени. Идентификаторы строки и
частицы сохраняют трассируемость до входного файла. Никакой подгонки к ответу
ОМ на этом шаге нет.

### Задания

1. Сравнить истинную длину шага с хордой.
2. Повторить PCA без фотонных весов и измерить поворот оси.
3. Для мюонного файла сопоставить временной диапазон и продольный размер.
